# Regression Model Accuracy Testing
___

Import pandas and the three regression classes.

In [ ]:
import pandas as pd
from regression import BivariableLinearRegression, MultipleLinearRegression, LogisticLinearRegression

Load the dataset and drop any rows with missing values.

In [ ]:
df = pd.read_csv('StudentPerformanceFactors.csv').dropna()
df.head()

Categorical columns need to be encoded as integers so the models can work with them. Each unique string value is mapped to a number.

In [ ]:
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype('category').cat.codes

df.head()

Shuffle and split into 80% training and 20% testing.

In [ ]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

split = int(len(df) * 0.8)
train = df.iloc[:split]
test  = df.iloc[split:]

print(f'Train: {len(train)} rows, Test: {len(test)} rows')

___
## BivariableLinearRegression
Predicting `Exam_Score` from `Hours_Studied` only.

In [ ]:
model_biv = BivariableLinearRegression(
    dataset=train[['Hours_Studied', 'Exam_Score']].reset_index(drop=True)
)

predictions = model_biv.predict(test['Hours_Studied'].values)
actual      = test['Exam_Score'].values

mae  = sum(abs(predictions - actual)) / len(actual)
rmse = (sum((predictions - actual) ** 2) / len(actual)) ** 0.5

print(f'MAE  : {mae:.4f}')
print(f'RMSE : {rmse:.4f}')

___
## MultipleLinearRegression
Predicting `Exam_Score` from all features.

In [ ]:
features = [c for c in df.columns if c != 'Exam_Score']

model_multi = MultipleLinearRegression(
    dataset=train[features + ['Exam_Score']].reset_index(drop=True)
)

predictions = model_multi.predict(test[features].values).flatten()
actual      = test['Exam_Score'].values

mae  = sum(abs(predictions - actual)) / len(actual)
rmse = (sum((predictions - actual) ** 2) / len(actual)) ** 0.5

print(f'MAE  : {mae:.4f}')
print(f'RMSE : {rmse:.4f}')

___
## LogisticLinearRegression
Classifying whether a student passes (Exam_Score >= 70) or fails. `Exam_Score` is excluded from the features to avoid leaking the answer into the model.

In [ ]:
df_log = df.copy()
df_log['Pass'] = (df_log['Exam_Score'] >= 70).astype(int)

log_features = [c for c in features if c != 'Exam_Score']

train_log = df_log[log_features + ['Pass']].iloc[:split]
test_log  = df_log[log_features + ['Pass']].iloc[split:]

model_log = LogisticLinearRegression(
    dataset=train_log.reset_index(drop=True)
)

predictions = model_log.predict(test_log[log_features].values).flatten()
actual      = test_log['Pass'].values

correct  = sum(predictions == actual)
accuracy = correct / len(actual) * 100

print(f'Correct : {correct} / {len(actual)}')
print(f'Accuracy: {accuracy:.1f}%')